# Softmax Function
The softmax function converts a vector of raw scores into a probability distribution — used for multiclass classification, unlike sigmoid (binary) or ReLU (hidden layers).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Dense
from tensorflow.keras import Sequential
from sklearn.datasets import make_blobs

In [2]:
def my_softmax(z):
    ez = np.exp(z)
    sm = ez/np.sum(ez)
    return(sm)

### The obvious organization
The model below uses softmax directly as the final layer's activation, with SparseCategoricalCrossentropy as the loss.

In [5]:
centers = [[-5, 2], [-2, -2], [1, 2], [5, -2]]
X_train, y_train = make_blobs(n_samples=2000, centers=centers, cluster_std=1.0, random_state=30)

In [10]:
model = Sequential([
    Dense(25,activation='relu'),
    Dense(15,activation='relu'),
    Dense(4,activation='softmax')

    
])

model.compile(
    loss= tf.keras.losses.SparseCategoricalCrossentropy(),
    optimizer = tf.keras.optimizers.Adam(0.001)     
)

model.fit(X_train,y_train,epochs =10)

Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 0.8207
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4111
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1821
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1010
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0727
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0593
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0514
Epoch 8/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0459
Epoch 9/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0422
Epoch 10/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0388


### Obvious model output
Since softmax is built into the final layer, predictions are already a probability distribution.

In [11]:
p_nonpreferred = model.predict(X_train)
print(p_nonpreferred[:2])
print("largest value", np.max(p_nonpreferred), "smallest value", np.min(p_nonpreferred))

63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
[[6.2506166e-03 1.5862235e-03 9.7367835e-01 1.8484807e-02]
 [9.9617320e-01 3.6899559e-03 6.3564323e-05 7.3336647e-05]]
largest value 0.99999976 smallest value 1.435767e-09


### Preferred organization
More numerically stable results are obtained by using a linear output layer and letting the loss function handle softmax internally via from_logits=True.

In [12]:
preferred_model = Sequential([
    Dense(25, activation='relu'),
    Dense(15, activation='relu'),
    Dense(4, activation='linear')
])
preferred_model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(0.001),
)
preferred_model.fit(X_train, y_train, epochs=10)

Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 1.1322
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.5361
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1973
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0953
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0668
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0544
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0481
Epoch 8/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0436
Epoch 9/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0397
Epoch 10/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0363


### Output handling
The preferred model's raw outputs are not probabilities — apply softmax separately to convert them.

In [13]:
p_preferred = preferred_model.predict(X_train)
print(f"two example output vectors:\n {p_preferred[:2]}")
print("largest value", np.max(p_preferred), "smallest value", np.min(p_preferred))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
two example output vectors:
 [[-2.8828168 -1.0675274  3.500821  -0.7983819]
 [ 7.1210456  1.3742646 -4.765891  -9.69061  ]]
largest value 11.809685 smallest value -13.539534


In [16]:
#apply softmax manually, after raw output

sm_preferred = tf.nn.softmax(p_preferred).numpy()  # .numpy() converts tensorflow array to simple array
print(f"two example output vectors:\n {sm_preferred[:2]}")
print("largest value", np.max(sm_preferred), "smallest value", np.min(sm_preferred))

two example output vectors:
 [[1.6467393e-03 1.0115681e-02 9.7499770e-01 1.3239862e-02]
 [9.9681014e-01 3.1828575e-03 6.8577501e-06 4.9819963e-08]]
largest value 0.9999993 smallest value 1.8242833e-11
